In [106]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Optional , Annotated 
# from langgraph.graph.message import add_messages ,BaseMessage
# from langchain_core.messages import HumanMessage, AIMessage 
from langgraph.checkpoint.memory import InMemorySaver

In [107]:
class ExpenseSchema(TypedDict):
    expense_description: str
    expense_amount: float
    needs_review: Optional[bool]
    approved: Optional[bool]
    outcome_message: Optional[str]

In [108]:
REVIEW_THRESHOLD = 100

def review_expense(state: ExpenseSchema) -> dict:
    amount = state["expense_amount"]
    needs_review = amount > REVIEW_THRESHOLD
    return {"needs_review": needs_review}

In [109]:
from langgraph.types import interrupt

def human_review(state: ExpenseSchema) -> dict:
    description = state["expense_description"]
    amount = state["expense_amount"]

    human_decision = interrupt({
        "question": "Approve this expense?",
        "description": description,
        "amount": amount
    })

    return {"approved": human_decision}

In [110]:
def finalize_decision(state: ExpenseSchema) -> dict:
    needs_review = state["needs_review"]
    approved = state.get("approved")

    if not needs_review:
        outcome = f"Auto-approved: ${state['expense_amount']} for '{state['expense_description']}' (below review threshold)."
    elif approved:
        outcome = f"Approved by reviewer: ${state['expense_amount']} for '{state['expense_description']}'."
    else:
        outcome = f"Rejected by reviewer: ${state['expense_amount']} for '{state['expense_description']}'."

    return {"outcome_message": outcome}

In [111]:
REVIEW_THRESHOLD = 100

graph = StateGraph(ExpenseSchema)

graph.add_node("review_expense", review_expense)
graph.add_node("human_review", human_review)
graph.add_node("finalize_decision", finalize_decision)

graph.add_edge(START, "review_expense")

def route_after_review(state: ExpenseSchema) -> str:
    if state["needs_review"]:
        return "needs_review"
    else:
        return "auto_approved"

graph.add_conditional_edges(
    "review_expense",
    route_after_review,
    {
        "needs_review": "human_review",
        "auto_approved": "finalize_decision"
    }
)

graph.add_edge("human_review", "finalize_decision")
graph.add_edge("finalize_decision", END)

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [112]:
config = {"configurable": {"thread_id": "1"}}

result = app.invoke(
    {"expense_description": "team dinner", "expense_amount": 250},
    config=config
)
print(result)

{'expense_description': 'team dinner', 'expense_amount': 250, 'needs_review': True, '__interrupt__': [Interrupt(value={'question': 'Approve this expense?', 'description': 'team dinner', 'amount': 250}, id='899a3c4cce987b88ad810fdad24edc94')]}


In [113]:
app.get_state(config=config)

StateSnapshot(values={'expense_description': 'team dinner', 'expense_amount': 250, 'needs_review': True}, next=('human_review',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19adb8-a270-6227-8001-9008e611b71f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-18T08:05:13.152362+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19adb8-a26d-6b0d-8000-bbd2d03a800f'}}, tasks=(PregelTask(id='5d240c18-56dd-b010-d6f0-cf89ff82ed5f', name='human_review', path=('__pregel_pull', 'human_review'), error=None, interrupts=(Interrupt(value={'question': 'Approve this expense?', 'description': 'team dinner', 'amount': 250}, id='899a3c4cce987b88ad810fdad24edc94'),), state=None, result=None),), interrupts=(Interrupt(value={'question': 'Approve this expense?', 'description': 'team dinner', 'amount': 250}, id='899a3c4cce987b88ad810fdad24edc94'),))

In [114]:
from langgraph.types import Command

result = app.invoke(Command(resume=True), config=config)
print(result)

{'expense_description': 'team dinner', 'expense_amount': 250, 'needs_review': True, 'approved': True, 'outcome_message': "Approved by reviewer: $250 for 'team dinner'."}


In [115]:
app.get_state(config=config)

StateSnapshot(values={'expense_description': 'team dinner', 'expense_amount': 250, 'needs_review': True, 'approved': True, 'outcome_message': "Approved by reviewer: $250 for 'team dinner'."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19adb8-a2e9-6a06-8003-7449cbcef6d9'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-18T08:05:13.202125+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19adb8-a2e9-6a05-8002-4a26ccff975d'}}, tasks=(), interrupts=())